# Reddit Kolkata analytics

Runs SQL queries directly against the partitioned Parquet files under `reddit_kolkata/parquet/` via DuckDB -- no data is imported into a database file. Set the parameters in the cell below, then run all cells.

Before running this notebook for the first time (or after new data is collected), convert the latest `processed/*.csv` to Parquet:
```
python -m reddit_kolkata.collection_code.convert_to_parquet --dataset both
```

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REDDIT_KOLKATA_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PARQUET_DIR = REDDIT_KOLKATA_DIR / "parquet"
EXPORT_DIR = REDDIT_KOLKATA_DIR / "processed" / "notebook_exports"
POSTS_PARQUET_DIR = PARQUET_DIR / "posts"
COMMENTS_PARQUET_DIR = PARQUET_DIR / "comments"

In [ ]:
# ===== PARAMETERS =====
# Edit these and re-run the notebook.
SUBREDDIT = "kolkata"                          # a subreddit slug present in the data, or None for "all subreddits"
START_DATE = "2025-04-01"                       # inclusive, YYYY-MM-DD
END_DATE = "2025-12-31"                         # inclusive, YYYY-MM-DD
KEYWORDS = ["BJP", "TMC", "election"]           # case-insensitive keyword search list

In [ ]:
import duckdb

con = duckdb.connect()  # in-memory connection only -- no .duckdb file is ever created

con.execute(f"""
    CREATE OR REPLACE VIEW posts AS
    SELECT * FROM read_parquet('{POSTS_PARQUET_DIR}/**/*.parquet', hive_partitioning=1)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW comments AS
    SELECT * FROM read_parquet('{COMMENTS_PARQUET_DIR}/**/*.parquet', hive_partitioning=1)
""")

con.execute("SELECT subreddit, COUNT(*) AS rows FROM posts GROUP BY subreddit ORDER BY subreddit").df()

In [ ]:
def export_to_csv(df, filename: str) -> Path:
    """Export a query result DataFrame to processed/notebook_exports/<filename>."""
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = EXPORT_DIR / filename
    df.to_csv(out_path, index=False)
    print(f"Exported {len(df)} rows -> {out_path}")
    return out_path

## (a) Total number of posts in each subreddit

In [ ]:
total_posts = con.execute("""
    SELECT subreddit, COUNT(*) AS total_posts
    FROM posts
    GROUP BY subreddit
    ORDER BY subreddit
""").df()
total_posts

## (b) Monthly number of posts in each subreddit

In [ ]:
monthly_posts = con.execute("""
    SELECT subreddit, strftime(date_posted, '%Y-%m') AS month, COUNT(*) AS num_posts
    FROM posts
    GROUP BY subreddit, strftime(date_posted, '%Y-%m')
    ORDER BY subreddit, month
""").df()
monthly_posts

## (c) Number of unique authors in each subreddit

Counts distinct authors across posts *and* comments per subreddit, matching the definition already used by `collection_code/validate.py::validate_subreddit`.

In [ ]:
unique_authors = con.execute("""
    SELECT subreddit, COUNT(DISTINCT author) AS unique_authors
    FROM (
        SELECT subreddit, author FROM posts WHERE author IS NOT NULL
        UNION
        SELECT subreddit, author FROM comments WHERE author IS NOT NULL
    ) combined
    GROUP BY subreddit
    ORDER BY subreddit
""").df()
unique_authors

## (d) Number of authors who posted in both subreddits

The generalized query below works for any number of subreddits (no hardcoded names), so it keeps working after a third subreddit is added. A literal two-subreddit `INTERSECT` form is shown underneath as the direct equivalent for exactly two subreddits.

In [ ]:
authors_in_all_subreddits = con.execute("""
    SELECT author, COUNT(DISTINCT subreddit) AS subreddits_posted_in
    FROM posts
    WHERE author IS NOT NULL
    GROUP BY author
    HAVING COUNT(DISTINCT subreddit) = (SELECT COUNT(DISTINCT subreddit) FROM posts)
    ORDER BY author
""").df()
print(f"Authors active in every currently-present subreddit: {len(authors_in_all_subreddits)}")
authors_in_all_subreddits.head(20)

In [ ]:
# Literal two-subreddit INTERSECT form -- assumes exactly two subreddits are present.
# If a third subreddit is added later, use the generalized GROUP BY/HAVING query above instead.
subs = con.execute("SELECT DISTINCT subreddit FROM posts ORDER BY subreddit").df()["subreddit"].tolist()

if len(subs) == 2:
    both_df = con.execute(
        """
        SELECT author FROM posts WHERE subreddit = ? AND author IS NOT NULL
        INTERSECT
        SELECT author FROM posts WHERE subreddit = ? AND author IS NOT NULL
        """,
        [subs[0], subs[1]],
    ).df()
    print(f"Authors who posted in both {subs[0]} and {subs[1]}: {len(both_df)}")
else:
    both_df = None
    print(f"{len(subs)} subreddits present ({subs}) -- INTERSECT form only applies to exactly 2; use the generalized query above.")
both_df

## (e) Top 20 highest-scoring posts in each subreddit

In [ ]:
top_scoring = con.execute("""
    SELECT subreddit, post_id, title, author, score, num_comments, date_posted, permalink
    FROM posts
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY score DESC) <= 20
    ORDER BY subreddit, score DESC
""").df()
top_scoring

## (f) Top 20 most-commented posts in each subreddit

In [ ]:
top_commented = con.execute("""
    SELECT subreddit, post_id, title, author, num_comments, score, date_posted, permalink
    FROM posts
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY num_comments DESC) <= 20
    ORDER BY subreddit, num_comments DESC
""").df()
top_commented

## (g) Posts containing specified keywords within a date range

Uses `SUBREDDIT`, `START_DATE`, `END_DATE`, `KEYWORDS` from the parameters cell above. Every user-supplied value (keywords, dates, subreddit) is bound through DuckDB's `?` parameter list -- never string-interpolated into the SQL text -- so free-text keyword input can't break or inject into the query; only the *count* of `?` placeholders depends on `len(KEYWORDS)`.

In [ ]:
def search_posts_by_keywords(subreddit, start_date, end_date, keywords):
    clauses, params = [], [start_date, end_date]
    for kw in keywords:
        clauses.append("(title ILIKE ? OR post_text ILIKE ?)")
        pattern = f"%{kw}%"
        params.extend([pattern, pattern])
    where_keywords = " OR ".join(clauses) if clauses else "FALSE"

    sql = f"""
        SELECT subreddit, post_id, title, author, date_posted, score, num_comments, permalink
        FROM posts
        WHERE date_posted >= ?
          AND date_posted < CAST(? AS DATE) + INTERVAL 1 DAY
          AND ({where_keywords})
    """
    if subreddit:
        sql += " AND subreddit = ?"
        params.append(subreddit)
    sql += " ORDER BY date_posted"

    return con.execute(sql, params).df()

keyword_results = search_posts_by_keywords(SUBREDDIT, START_DATE, END_DATE, KEYWORDS)
print(f"{len(keyword_results)} posts matched {KEYWORDS} between {START_DATE} and {END_DATE} (subreddit={SUBREDDIT!r})")
keyword_results.head(20)

## (h) Top 20 most frequently linked external domains in each subreddit

In [ ]:
top_domains = con.execute("""
    SELECT subreddit, external_domain, COUNT(*) AS link_count
    FROM posts
    WHERE external_domain IS NOT NULL AND external_domain <> ''
    GROUP BY subreddit, external_domain
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subreddit ORDER BY COUNT(*) DESC) <= 20
    ORDER BY subreddit, link_count DESC
""").df()
top_domains

## Export example

Any of the DataFrames above can be exported the same way; the keyword-search result is used here as the demonstration.

In [ ]:
export_to_csv(keyword_results, "keyword_search_results.csv")